# Cambodia Road Crash Portal: analysis notebook

This notebook reproduces every number on the portal's **Findings** page, starting from the raw files, and shows where each course
week appears. It calls the same functions as the portal (`src/pipeline.py`, `src/findings.py`), so the two cannot disagree.

**Research question.** Where and when do the road crashes reported in Khmer-language news cluster, how much of the real burden does
that coverage capture, and how does Cambodia's road death rate compare with ASEAN?

| Hypothesis | Test |
|---|---|
| H1a. Crashes reported outside Phnom Penh are more often fatal | One-sided two-proportion z-test |
| H1b. The capital is over-reported for its population | Chi-square of reports against population shares |
| H2. Cambodia's road death rate rose 2000-2019 while most ASEAN members' fell | Count of members whose WHO-modelled rate fell |

Run with **Kernel > Restart & Run All**. Random seed: 42 everywhere.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.io as pio

from src import charts, config, fetch
from src import pipeline as P
from src.findings import compute_findings
from src.names import province_table

pio.renderers.default = "notebook_connected"
np.random.seed(config.RANDOM_STATE)
pd.set_option("display.max_columns", 30, "display.width", 160)
print("Project root:", ROOT)

Project root: /home/virak/Desktop/modeling


## Week 1. One preprocessing path from each raw source

| Source | Raw file (as retrieved) | Step | Analysis table |
|---|---|---|---|
| Khmer news | team workbook -> `data/raw/news/Road_Accidents_Extracted.xlsx` (local only, copyrighted text) | `src/extract_accidents.py`, then `build_public_crash_table` drops text and sub-province places | `data/processed/crashes_public.csv` |
| ODC census 2019 + boundaries | `data/raw/odc/final_census2019_en.geojson` | `build_census_and_boundaries` (name matching, polygon simplification) | `census2019_province.csv`, `khm_provinces_census2019.geojson` |
| World Bank | `data/raw/worldbank/*.json` | `build_worldbank` (long to wide) | `worldbank_asean.csv` |
| Official totals | `data/reference/official_totals.csv`: National Road Safety Committee annual figures 2020-2024, typed in from the ministry or state-media page named per year | none | same |

`build_all` runs every step. Raw files are only read, never edited. The fetch step (`python -m src.fetch`) is not re-run here so that the
notebook gives the same numbers offline; the fetch log shows when each raw file was pulled.

In [2]:
built = P.build_all()
display(fetch.read_fetch_log().groupby("source").tail(1)[["retrieved_at_utc", "source", "rows", "status"]])

Public crash table rebuilt from the raw workbook: 1074 rows


Census: 25 provinces; boundaries: 96 KB
World Bank: 396 country-years, 11 countries


Rare event: base rate 14.4% in 2024, precision 0.35, recall 0.73


,retrieved_at_utc,source,rows,status
6,2026-11-08T08:00:48Z,World Bank SH.STA.TRAF.P5,220,ok
7,2026-11-08T08:00:49Z,World Bank SP.POP.TOTL,396,ok
8,2026-11-08T08:00:50Z,World Bank SP.URB.TOTL.IN.ZS,396,ok
9,2026-11-08T08:00:51Z,World Bank NY.GDP.PCAP.CD,396,ok
10,2026-11-08T08:01:51Z,ODC GeoServer census 2019 provinces,25,ok
11,2026-11-08T08:01:52Z,ODC CKAN package population-census-2019-final-...,1,ok


In [3]:
public = P.load_public_crashes()
crashes = P.prepare_crashes(public)
census, worldbank, official = P.load_census(), P.load_worldbank(), P.load_official()
funnel = pd.Series({
    "Rows in the news workbook": len(public),
    "Crash reports": int((public.record_type == "Crash report").sum()),
    "...first record of each crash": int(((public.record_type == "Crash report") & (public.is_primary_record == 1)).sum()),
    "...text appears original (analysis_ready)": int((public.analysis_ready == 1).sum()),
    "...with a known year in 2018-2024 (analysis window)": int(crashes.year.between(2018, 2024).sum()),
}, name="rows")
display(funnel.to_frame())
display(pd.read_csv(config.DATA_DICTIONARY))

,rows
Rows in the news workbook,1074
Crash reports,943
...first record of each crash,842
...text appears original (analysis_ready),814
...with a known year in 2018-2024 (analysis window),736


,table,column,type,unit,meaning,source
0,crashes_public,origin,text,-,"news dataset, or upload: <file name>",pipeline
1,crashes_public,doc_id,integer,-,Row number in the team's source workbook,news dataset
2,crashes_public,event_id,text,-,Same ID = same crash reported more than once,src/extract_accidents.py
3,crashes_public,analysis_ready,0/1,-,"1 = single crash in Cambodia, first record of ...",src/extract_accidents.py
4,crashes_public,record_type,category,-,"Crash report, Statistics/aggregate, Follow-up/...",src/extract_accidents.py
5,crashes_public,text_provenance,category,-,"Appears original, Suspected AI summary/rewrite...",src/extract_accidents.py
6,crashes_public,incident_date,date,day,Date of the crash; publication date used as a ...,news text
7,crashes_public,incident_date_source,category,-,"Article text, Publication date (proxy), Partia...",src/extract_accidents.py
8,crashes_public,year / month,integer,-,Calendar parts of incident_date,pipeline
9,crashes_public,time_of_day,category,-,"Morning (05-11), Afternoon (12-16), Evening (1...",news text


## Week 2. Missing values

Blank, `Unknown`, `Not stated` and `Unclassified` all count as missing in the map below. Reasons:
province is missing when the article names only a road; severity when it reports no outcome; time when no clock time or only an
hour without morning or evening is given. None of these are filled: an `Unknown` category is kept, because guessing would invent
night crashes or fatalities. The one fill is shown before and after: deaths = 1 when an article says someone died without a number.

In [4]:
cols = ["incident_date", "province", "severity", "time_of_day", "road_type", "road_name", "collision_type", "stated_factors"]
known_year = crashes.dropna(subset=["year"])
missing_map = P.missingness_by_year(known_year[known_year.year.between(2016, 2024)], cols)
charts.missingness_heatmap(missing_map, "Missingness map, news crashes: road names and factors are mostly absent").show()

assumed = crashes.casualty_note.fillna("").str.contains("assumed 1")
print(f"Fill: {assumed.sum()} fatal crashes had no number of deaths. Before: blank. After: 1.")
print(f"Deaths with the fill: {crashes.deaths.sum():.0f}; without it: {crashes.deaths.sum() - assumed.sum():.0f}")
fill = P.imputation_before_after(crashes)
charts.before_after_chart(fill, f"Before and after the one fill on the same axes: {fill.attrs['filled']} fatal crashes "
                                "with no death count became 1").show()

wb_missing = worldbank.pivot(index="country", columns="year", values="road_deaths_per_100k").isna().mul(100)
charts.missingness_heatmap(wb_missing, "Missingness map, World Bank road mortality: the WHO series covers 2000-2019 only").show()
print("Official totals missing:", official[["official_deaths", "official_crashes"]].isna().mean().round(2).to_dict())

Fill: 92 fatal crashes had no number of deaths. Before: blank. After: 1.
Deaths with the fill: 538; without it: 446


Official totals missing: {'official_deaths': 0.0, 'official_crashes': 0.0}


## Week 3. Outliers: flagged, never deleted

Monthly counts use a modified z-score (median and MAD, |z| > 3.5) because the spikes themselves would inflate a standard deviation.
Deaths and injuries per crash use 3 x IQR fences.

In [5]:
series = P.monthly_series(crashes)
spikes = P.robust_outliers(series)
display(spikes[spikes.flagged])
fatal = crashes[crashes.deaths > 0]
low, high = P.iqr_fences(fatal.deaths)
print(f"Deaths per fatal crash: fence {high:g}; {(fatal.deaths > high).sum()} crashes above it (bus and truck crashes, kept).")
injured = crashes[crashes.injuries_total > 0]
low_i, high_i = P.iqr_fences(injured.injuries_total)
print(f"Injuries per injury crash: fence {high_i:g}; {(injured.injuries_total > high_i).sum()} above it (kept, flagged for review).")

,value,robust_z,flagged
2018-08-01,25.0,4.496667,True
2021-10-01,42.0,8.318833,True
2021-11-01,46.0,9.218167,True
2021-12-01,55.0,11.241667,True
2022-01-01,44.0,8.768500,True
2024-09-01,28.0,5.171167,True
2024-10-01,40.0,7.869167,True
2024-11-01,40.0,7.869167,True


Deaths per fatal crash: fence 5; 16 crashes above it (bus and truck crashes, kept).
Injuries per injury crash: fence 23; 30 above it (kept, flagged for review).


**Explanation.** Every flagged month (Aug 2018, Oct 2021 to Jan 2022, Sep to Nov 2024) matches a scraping batch, not an event on the
roads. Deleting them would hide the main weakness of this source, so they stay in, and the forecast below carries their noise.

## Week 4. Normalisation

Maps never show counts. Reports and deaths are divided by person-years: census 2019 population moved to each year with the national
growth rate from the World Bank. Fatal shares are shown only where a province has at least 10 reports with known severity.
The model and PCA use `StandardScaler`, because logistic regression and PCA are both sensitive to scale; flags stay 0/1.

In [6]:
panel = P.province_panel(crashes[crashes.year.between(2018, 2024)], census, worldbank, 2018, 2024)
display(panel[["province", "reports", "population_2019", "person_years", "reports_per_100k_yr", "deaths_per_100k_yr",
               "fatal_share_pct", "report_share_pct", "population_share_pct"]].sort_values("reports_per_100k_yr", ascending=False).round(2))

,province,reports,population_2019,person_years,reports_per_100k_yr,deaths_per_100k_yr,fatal_share_pct,report_share_pct,population_share_pct
17,Preah Sihanouk,76,310072,2233546.02,3.40,1.43,25.37,10.51,1.99
11,Phnom Penh,407,2281951,16437609.88,2.48,0.57,23.92,56.29,14.67
22,Kep,4,42665,307329.40,1.30,1.30,NaN,0.55,0.27
8,Koh Kong,6,125902,906911.66,0.66,0.66,NaN,0.83,0.81
10,Mondulkiri,4,92213,664239.21,0.60,1.05,NaN,0.55,0.59
14,Pursat,16,419952,3025046.18,0.53,0.46,60.00,2.21,2.70
16,Siem Reap,34,1014234,7305846.10,0.47,0.71,75.00,4.70,6.52
1,Battambang,32,997169,7182921.55,0.45,0.47,59.38,4.43,6.41
4,Kampong Speu,28,877523,6321073.83,0.44,0.65,53.85,3.87,5.64
15,Ratanakiri,6,217453,1566382.27,0.38,0.06,NaN,0.83,1.40


## Weeks 5 and 9. Encoding and data integration

In [7]:
display(province_table())
print("Grain and key of each source:")
display(pd.DataFrame([
    ("News crashes", "one crash", "event_id; province name -> pro_code"),
    ("Census 2019", "one province", "pro_code"),
    ("Province boundaries", "one province polygon", "pro_code (properties.pro_code)"),
    ("World Bank", "one country-year", "iso3 + year"),
    ("Official totals", "one year", "year"),
], columns=["source", "grain", "key"]))
display(pd.read_csv(config.JOIN_LOG))
rare = json.loads(config.RARE_EVENT_JSON.read_text())
print("One-hot widths added by each categorical column:", rare["encoder_widths"])
print("Total model columns after encoding:", rare["n_features_after_encoding"])

,pro_code,iso_3166_2,province,name_census,name_khmer,aliases
0,1,KH-1,Banteay Meanchey,Banteay Meanchey,បន្ទាយមានជ័យ,Banteay Mean Chey
1,2,KH-2,Battambang,Battambang,បាត់ដំបង,Batdambang
2,3,KH-3,Kampong Cham,Kampong Cham,កំពង់ចាម,Kompong Cham
3,4,KH-4,Kampong Chhnang,Kampong Chhnang,កំពង់ឆ្នាំង,Kompong Chhnang
4,5,KH-5,Kampong Speu,Kampong Speu,កំពង់ស្ពឺ,Kompong Speu|Kampong Spoe
5,6,KH-6,Kampong Thom,Kampong Thom,កំពង់ធំ,Kompong Thom
6,7,KH-7,Kampot,Kampot,កំពត,Kampaot
7,8,KH-8,Kandal,Kandal,កណ្តាល,កណ្ដាល
8,9,KH-9,Koh Kong,Koh Kong,កោះកុង,Kaoh Kong
9,10,KH-10,Kratie,Kratie,ក្រចេះ,Kracheh|Kratié


Grain and key of each source:


,source,grain,key
0,News crashes,one crash,event_id; province name -> pro_code
1,Census 2019,one province,pro_code
2,Province boundaries,one province polygon,pro_code (properties.pro_code)
3,World Bank,one country-year,iso3 + year
4,Official totals,one year,year


,step,rows_left,rows_right,rows_after,unmatched_left,key,grain
0,News crashes x name-matching table,814,25,794,0,province name -> pro_code,crash
1,Unlocated crashes (no province in text),814,0,794,20,-,crash
2,Census layer x name-matching table,25,25,25,0,census name -> pro_code,province
3,Province crash totals x census population,25,25,25,0,pro_code,province (2018-2024)
4,Census population x World Bank national popula...,25,36,175,0,year (national growth factor),province-year
5,Annual news totals x official totals x WHO mod...,13,5,5,0,year,year


One-hot widths added by each categorical column: {'time_of_day': 5, 'day_type': 2, 'season': 2, 'province_group': 6, 'road_type': 6}
Total model columns after encoding: 46


Province is encoded twice: as the official code and ISO 3166-2 code (`KH-12`) for joins and maps, and as one-hot columns of a
top-5 + Other grouping in the model. One-hot was chosen over target encoding because the target is rare (about 10%): target means
for small provinces would be mostly noise and would leak the label if not cross-fitted. The cost: the 20 smaller provinces share one
`Other` column.

## Week 6. Feature engineering, each a claim about the world

The claims are written in the data dictionary above. The main ones: reports per resident remove province size; the fatal share is
insensitive to how many crashes journalists cover; change on a year earlier compares like seasons; a 3-month rolling share smooths
single large crashes; the rainy season (May-October) changes road surfaces; buses and trucks carry and hit more people.

## Week 8. Feature selection: a correlation rule, then PCA

Rule set before looking at the data: of any two indicators with |Spearman rho| >= 0.8, the one listed later in `PCA_INDICATORS` is
dropped; the list is ordered by completeness (reports count every located crash, deaths need the article to state a number).
PCA then runs on what the rule keeps, with the variance share before the rule next to it, so the cost of the rule is visible.

In [8]:
pca = P.province_pca(panel)
print(f"Rule |rho| >= {pca['limit']}: kept {len(pca['kept'])} of {len(P.PCA_INDICATORS)} indicators; dropped {[d['dropped'] for d in pca['dropped']]}")
for d in pca["dropped"]:
    print(f"  {d['dropped']}: rho {d['rho']:.2f} with {d['kept']}")
corr = pca["correlation"]
strongest = corr.where(~np.eye(len(corr), dtype=bool)).abs().stack().idxmax()
heat_title = (f"One pair above |rho| = {pca['limit']}: {charts.indicator_label(strongest[0])} and "
              f"{charts.indicator_label(strongest[1])} (rho {corr.loc[strongest]:.2f})" if pca["dropped"]
              else f"No pair reaches |rho| = {pca['limit']}: strongest is {corr.loc[strongest]:.2f}")
charts.correlation_heatmap(corr, heat_title, [d["dropped"] for d in pca["dropped"]]).show()
charts.variance_chart(pca["variance_share"], pca["variance_share_all"], len(pca["kept"]), len(P.PCA_INDICATORS),
                      f"PC1 keeps {pca['variance_share'][0]:.0%} of the variance after the rule "
                      f"({pca['variance_share_all'][0]:.0%} before)").show()
print(f"PCA over {pca['provinces']} provinces with 10+ reports. Variance kept:", np.round(pca["variance_share"], 3))
display(pca["loadings"].round(2))
display(pca["index"].sort_values(ascending=False).round(2).to_frame())


Rule |rho| >= 0.8: kept 7 of 8 indicators; dropped ['deaths_per_100k_yr']
  deaths_per_100k_yr: rho 0.86 with reports_per_100k_yr


PCA over 11 provinces with 10+ reports. Variance kept: [0.55  0.153 0.127]


,PC1,PC2,PC3
reports_per_100k_yr,0.46,-0.05,0.04
fatal_share_pct,-0.44,0.08,-0.25
density_per_km2,0.38,-0.22,-0.44
household_size,0.29,0.26,0.76
national_road_share_pct,-0.37,-0.18,0.30
night_share_pct,0.47,-0.05,-0.14
motorcycle_share_pct,0.03,0.92,-0.25


,PC1 index
province,
Phnom Penh,4.61
Preah Sihanouk,3.32
Kandal,-0.05
Battambang,-0.37
Siem Reap,-0.40
Kampong Speu,-0.63
Banteay Meanchey,-0.69
Pursat,-1.11
Kampot,-1.18


PC1 contrasts provinces with many reports per resident, night crashes and high density (the capital and Preah Sihanouk) against
provinces where reports are rarer, more often fatal and on national roads. It reads as an *urban coverage* index, not a danger index,
so it is reported but not used as a model feature. The rule cost nothing: PC1 keeps 55% of the variance of the 7 kept
indicators against 53% of all 8, because deaths per 100,000 carried the same information as reports per 100,000 (rho 0.86).

## Week 11. Time series

Common frequency: every crash is placed in the month of its incident date (empty months count as 0); annual series (World Bank,
official totals) are joined by year, after summing months to years. Trend and season are separated with STL on log(1+x).
The split is in time: fit to Dec 2023, score the twelve 2024 months once, against a seasonal naive forecast (same month last year).

In [9]:
decomposition = P.decompose(series)
forecast = P.forecast_vs_naive(series)
charts.time_chart(series, decomposition, forecast, spikes, "Reported crashes per month with trend, forecast and naive baseline").show()
charts.seasonal_chart(decomposition, "Seasonal factors follow collection batches").show()
charts.yoy_chart(P.year_on_year(series).dropna(), "Change on the same month a year earlier").show()
display(forecast["test"].round(1))
print({k: round(v, 3) for k, v in forecast.items() if k.startswith(("mae", "mase"))})

,actual,model,low80,high80,seasonal_naive
2024-01-01,6.0,6.8,2.1,18.8,5.0
2024-02-01,10.0,2.3,0.0,11.2,0.0
2024-03-01,10.0,4.4,0.1,26.0,5.0
2024-04-01,8.0,3.4,0.0,27.3,3.0
2024-05-01,5.0,3.9,0.0,37.9,7.0
2024-06-01,12.0,9.3,0.1,98.4,4.0
2024-07-01,10.0,6.4,0.0,85.0,5.0
2024-08-01,18.0,7.4,0.0,113.8,7.0
2024-09-01,28.0,5.2,0.0,98.6,3.0
2024-10-01,40.0,6.3,0.0,135.9,2.0


{'mae_model': 10.774, 'mae_naive': 12.167, 'mase_model': 1.029, 'mase_naive': 1.162}


In [10]:
khm = worldbank[worldbank.iso3 == "KHM"].set_index("year").road_deaths_per_100k
annual = P.annual_trend_vs_naive(khm)
display(annual["test"].round(2))
print(f"Annual WHO series, 2015-2019 held out: trend MAE {annual['mae_trend']:.2f}, naive MAE {annual['mae_naive']:.2f}")

,actual,trend,naive
year,,,
2015,17.4,17.42,17.3
2016,17.8,17.40,17.3
2017,18.4,17.37,17.3
2018,19.0,17.34,17.3
2019,19.6,17.31,17.3


Annual WHO series, 2015-2019 held out: trend MAE 1.08, naive MAE 1.14


## Weeks 10 and 13. Rare-event task in one pipeline

In [11]:
display(pd.Series({k: v for k, v in rare.items() if not isinstance(v, (dict, list))}).to_frame("value"))
print("Candidates (5-fold CV PR-AUC on 2018-2023):", rare["candidates"])
print("Threshold by cost ratio (miss : false alarm):", rare["threshold_by_cost_ratio"])
costs = pd.read_csv(config.RARE_EVENT_COSTS)
charts.cost_curve(costs, rare["threshold"], f"Threshold {rare['threshold']:.2f} minimises 5 x misses + 1 x false alarms").show()

check = pd.DataFrame(rare["feature_check"])
print("Rule:", rare["feature_rule"])
print("Groups dropped:", rare["feature_groups_dropped"] or "none")
display(check.round(3))
carries = int((check.change.iloc[1:] < -check.fold_sd.iloc[0]).sum())
charts.ablation_chart(check, f"{carries} of {len(check) - 1} feature groups carry signal; {len(rare['feature_groups_dropped'])} "
                             "dropped under the rule").show()
coefs = pd.Series({r["feature"]: r["coefficient"] for r in rare["coefficients"]})
charts.coefficient_chart(coefs, f"'{charts.feature_label(coefs.idxmax())}' raises and '{charts.feature_label(coefs.idxmin())}' lowers "
                                "the odds most<br><sup>Largest coefficients of the final model; associations in reported crashes, not causes</sup>").show()

,value
trained_at_utc,2026-11-14 15:24 UTC
target,Crash with two or more deaths (multi-fatality)
train_years,2018-2023
test_year,2024
train_rows,477
test_rows,180
base_rate_train,0.09434
base_rate_test,0.144444
positives_test,26
chosen,"Class weights, no resampling"


Candidates (5-fold CV PR-AUC on 2018-2023): {'Class weights, no resampling': {'cv_pr_auc': 0.39181519837145473, 'C': 0.3}, 'Random oversampling inside folds': {'cv_pr_auc': 0.38995313189982983, 'C': 0.03}}
Threshold by cost ratio (miss : false alarm): {'2': 0.7, '5': 0.54, '10': 0.36}


Rule: A feature group is dropped only if the model does better without it by more than one fold standard deviation of the full model (5-fold CV PR-AUC on the training years). A loss, or a change inside that noise, keeps the group.
Groups dropped: none


,without,n_features,cv_pr_auc,fold_sd,change
0,nothing (all features),30,0.392,0.040,0.000
1,province,29,0.337,0.044,-0.055
2,road type,29,0.373,0.040,-0.019
3,time of day,29,0.399,0.044,0.007
4,day type,29,0.405,0.078,0.013
5,season,29,0.388,0.050,-0.004
6,number of vehicle types,29,0.399,0.051,0.008
7,intersection,29,0.389,0.043,-0.003
8,vehicle types,23,0.317,0.042,-0.074
9,stated factors,20,0.385,0.028,-0.006


Everything fitted (one-hot encoder, scaler, oversampler when used, logistic regression) is inside one `imblearn` pipeline, so
cross-validation refits it on each training fold and oversampling never sees validation rows. The threshold is chosen on
out-of-fold training predictions; 2024 is scored once. Accuracy is the wrong score here: never flagging anything would be right
for most 2024 crashes (see `accuracy_if_always_no`) and find no multi-fatality crash at all.

**Feature selection for the model** happens on the training years only. The chosen pipeline is refitted without each feature
group (province, road type, time of day, day type, season, vehicle types, stated factors, collision type, intersection, number of
vehicle types), and a pre-set rule drops a group only if the model does better without it by more than one fold standard deviation.
Province and vehicle types carry most of the signal; the calendar features add nothing measurable but are kept, because a change
inside the noise is not evidence either way. 2024 played no part in this choice.

## Hypothesis tests

In [12]:
window = crashes[crashes.year.between(2018, 2024)]
h1a = P.test_capital_vs_rest(window)
h1b = P.test_coverage_vs_population(panel)
h2 = P.test_asean_trend(worldbank)
display(pd.DataFrame({"H1a": pd.Series(h1a), "H1b": pd.Series(h1b), "H2": pd.Series(h2)}).round(4))

,H1a,H1b,H2
capital_population_share,NaN,14.6728,NaN
capital_rate,NaN,2.4760,NaN
capital_report_share,NaN,56.2932,NaN
chi2,NaN,1375.9848,NaN
fatal_share_inside,0.2392,NaN,NaN
fatal_share_outside,0.5533,NaN,NaN
khm_change,NaN,NaN,1.7
khm_rank,NaN,NaN,5.0
khm_rate,NaN,NaN,19.6
members,NaN,NaN,11.0


## Findings (identical to the portal's Findings page)

In [13]:
findings = compute_findings(crashes, census, worldbank, official)
for number, finding in enumerate(findings, 1):
    print(f"{number}. {finding.sentence}  [{finding.hypothesis or 'descriptive'}]")
    finding.figure.show()

1. Outside Phnom Penh 55% of reported crashes were fatal, against 24% inside the capital (2.3 times as often; z = 8.2, p < 0.001).  [H1a not rejected]


2. Phnom Penh holds 56% of reported crashes but 15% of the population, so it is reported 7.5 times as often per resident as the rest of the country (chi-square against population shares, p < 0.001).  [H1b not rejected]


3. In 2020-2024 the news sample recorded 362 road deaths against 7,951 in the National Road Safety Committee's totals, about 5%, and 615 of 14,983 recorded crashes (4%).  [descriptive]


4. Cambodia's WHO-modelled road death rate rose by 1.7 to 19.6 per 100,000 between 2000 and 2019, while 7 of the other 10 ASEAN members' rates fell; Cambodia ranks 5 of 11.  [H2 not rejected]


5. On the held-out 2024 months the Holt-Winters forecast beats 'same month last year' (mean absolute error 10.8 against 12.2 reports, +11%), but 8 of 84 months are flagged collection spikes.  [descriptive]


## Week 14. Ethics

* **Who is missing.** The sample holds roughly 5% of the deaths the National Road Safety Committee counted in 2020-2024 (and WHO's model puts the true toll at about twice the official count). Provinces with few reporters, non-injury crashes and
  people who died later in hospital are under-counted; foreign-language outlets and social media are absent.
* **Identification risk.** Articles name victims and publish plates. The public table drops all text, headlines, districts, communes,
  villages, kilometre markers and exact times; maps stop at the province and charts at the month.
* **Different treatment of groups.** The rare-event rule's recall differs between Phnom Penh and other provinces (see `recall_phnom_penh`
  above). Used to prioritise verification, it would skip the capital.
* **Modelled data.** WHO road mortality is an estimate whose quality differs by country.

## Weeks 7 and 15. Reproducibility

In [14]:
import importlib.metadata as md_
print({p: md_.version(p) for p in ["pandas", "numpy", "scikit-learn", "imbalanced-learn", "statsmodels", "plotly", "streamlit"]})
print("Random state:", config.RANDOM_STATE)

{'pandas': '3.1.1', 'numpy': '2.5.4', 'scikit-learn': '1.9.1', 'imbalanced-learn': '0.14.2', 'statsmodels': '0.15.0', 'plotly': '7.1.0', 'streamlit': '1.67.0'}
Random state: 42
